# Plot embeddings: GNN Baseline vs Split Architecture (not pretrained)

Compare graph-level embeddings from two **finetuned-from-scratch** models:
1. **GNN Baseline** — full bipartite GNN trained end-to-end (`GNNPolicy`)
2. **Split Architecture** — block-decomposed GNN with optional composer (`SplitBlockBiJepaPolicy`)

Both models are loaded from their `best.pt` checkpoints (no pretraining involved).

In [ ]:
from __future__ import annotations
import json
import os
import sys
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
from torch.utils.data import DataLoader
from torch_geometric.data import Batch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import configargparse
from tqdm.notebook import tqdm

from data.common import ProblemClass
from data.datasets import GraphDataset, PadFeaturesTransform
from data.split import SplitInstanceDataset, split_instance_collate, SplitInstanceBatch
from models.gnn import GNNPolicy
from models.split import SplitBlockBiJepaPolicy
from jobs.utils import set_all_seeds, device_from_args, build_dataloaders

## Configuration

Set checkpoint paths and shared hyperparameters below. Adjust `gnn_baseline_path` and `split_model_path` to point to your trained checkpoints.

In [ ]:
out_dir = Path("./results/finetuned_embeddings")
out_dir.mkdir(parents=True, exist_ok=True)

# ── Checkpoint paths (adjust these) ──────────────────────────────────
gnn_baseline_path = (
    "../experiments/finetune_CA_200/finetune_CA_200_gnn_baseline/"
    "kkt_gnn_finetuning_for_milp_experiments/"
    "run_gnn_policy-gv=2-lv=8-slice=512-point=13-lmbd=0.256-std=0.0"
    "-dim=128-l_mask=0.6-g_mask=0.05-l_e_mask=0.05-g_e_mask=0.6"
    "-dp=0.05-l_dim=128-conv=gatv2-heads=4_20260318_102025/best.pt"
)
split_model_path = (
    "../experiments/split_bijepa_finetuning/"
    "finetune_split_CA_200_RAW_h0/best.pt"
)

# ── Shared settings ──────────────────────────────────────────────────
SEED = 42
DEVICE_ID = "0"
BATCH_SIZE = 80
MAX_GRAPHS_PER_CLASS = 7500
POOLING = "concat"           # "concat" or "mean_all"
METHOD = "pca_tsne"          # "pca", "tsne", or "pca_tsne"
PCA_DIM = 50
PERPLEXITY = 30.0
PROBLEM_CLASSES = ["CA", "IS", "CFL", "SC"]

## Shared helpers

In [ ]:
set_all_seeds(SEED)
device = torch.device(f"cuda:{DEVICE_ID}" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


def infer_problem_class(sample_path: str) -> str:
    parts = Path(sample_path).parts
    for p in PROBLEM_CLASSES:
        if p in parts:
            return p
    sp = str(sample_path)
    for p in PROBLEM_CLASSES:
        if f"/{p}/" in sp or f"\\{p}\\" in sp:
            return p
    return "UNK"


def pooled_graph_embeddings(
    z: torch.Tensor, batch: Batch, pooling: str = "concat",
) -> torch.Tensor:
    """Pool node embeddings to graph-level for standard bipartite batches."""
    cb = batch.constraint_features_batch
    vb = batch.variable_features_batch
    B = int(batch.num_graphs)
    m_tot = int(cb.numel())
    ce = z[:m_tot]
    ve = z[m_tot:]

    def mean_by_graph(x, idx):
        out = x.new_zeros((B, x.size(-1)))
        out.index_add_(0, idx, x)
        counts = x.new_zeros((B,), dtype=torch.float32)
        counts.index_add_(0, idx, torch.ones(idx.size(0), device=x.device))
        return out / counts.clamp_min(1.0).unsqueeze(-1)

    cons_mean = mean_by_graph(ce, cb)
    var_mean = mean_by_graph(ve, vb)
    if pooling == "concat":
        return torch.cat([cons_mean, var_mean], dim=-1)
    elif pooling == "mean_all":
        return (cons_mean + var_mean) * 0.5
    else:
        raise ValueError(f"Unknown pooling: {pooling}")


def pooled_split_embeddings(
    model: SplitBlockBiJepaPolicy,
    batch: SplitInstanceBatch,
    pooling: str = "concat",
) -> Tuple[torch.Tensor, List[str]]:
    """
    Embed a SplitInstanceBatch and pool per-instance.
    Returns (graph_embeddings [B, D'], list of instance names).
    """
    composed = model._compose_instances(
        batch.instances, next(model.parameters()).device
    )
    embs = []
    for c_emb, v_emb in composed:
        c_mean = c_emb.mean(dim=0)
        v_mean = v_emb.mean(dim=0)
        if pooling == "concat":
            embs.append(torch.cat([c_mean, v_mean], dim=-1))
        else:
            embs.append((c_mean + v_mean) * 0.5)
    names = [inst.name for inst in batch.instances]
    return torch.stack(embs, dim=0), names


def pca_numpy(X: np.ndarray, n_components: int) -> np.ndarray:
    Xc = X - X.mean(axis=0, keepdims=True)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:n_components].T


def plot_2d(Y, labels, out_path, title):
    fig, ax = plt.subplots(figsize=(8, 6))
    classes = [c for c in PROBLEM_CLASSES if c in set(labels)]
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    cmap = {c: colors[i % len(colors)] for i, c in enumerate(classes)}
    for c in classes:
        idx = [i for i, l in enumerate(labels) if l == c]
        if not idx:
            continue
        pts = Y[idx]
        ax.scatter(pts[:, 0], pts[:, 1], s=10, alpha=0.8, label=c, c=cmap[c])
    ax.set_title(title)
    ax.set_xlabel("dim-1"); ax.set_ylabel("dim-2")
    ax.legend(loc="best", markerscale=2, frameon=True)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
    fig.tight_layout(); fig.savefig(out_path, dpi=200); plt.close(fig)


def plot_3d(Y, labels, out_path, title):
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")
    classes = [c for c in PROBLEM_CLASSES if c in set(labels)]
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    cmap = {c: colors[i % len(colors)] for i, c in enumerate(classes)}
    for c in classes:
        idx = [i for i, l in enumerate(labels) if l == c]
        if not idx:
            continue
        pts = Y[idx]
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=10, alpha=0.8, label=c, c=cmap[c])
    ax.set_title(title)
    ax.set_xlabel("dim-1"); ax.set_ylabel("dim-2"); ax.set_zlabel("dim-3")
    ax.legend(loc="best", markerscale=2, frameon=True)
    fig.tight_layout(); fig.savefig(out_path, dpi=200); plt.close(fig)


def reduce_and_plot(X, labels, tag, out_subdir):
    """Run PCA / t-SNE reduction and save plots."""
    X_std = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-6)

    if METHOD in ("pca", "pca_tsne"):
        Y2 = pca_numpy(X_std, 2)
        plot_2d(Y2, labels, str(out_subdir / f"{tag}_pca_2d.png"),
                f"PCA 2D — {tag} (pool={POOLING})")
        Y3 = pca_numpy(X_std, 3)
        plot_3d(Y3, labels, str(out_subdir / f"{tag}_pca_3d.png"),
                f"PCA 3D — {tag} (pool={POOLING})")
        print(f"  [OK] {tag} PCA plots saved")

    if METHOD in ("tsne", "pca_tsne"):
        pca_dim = int(min(PCA_DIM, X_std.shape[1], max(2, X_std.shape[0] - 1)))
        Xp = PCA(n_components=pca_dim, random_state=SEED).fit_transform(X_std)
        N = Xp.shape[0]
        perplex = max(5.0, min(PERPLEXITY, (N - 1) / 3.0))
        print(f"  [INFO] t-SNE: perplexity={perplex:.1f}, PCA pre-dim={pca_dim}")

        Y2 = TSNE(n_components=2, perplexity=perplex, init="pca",
                   learning_rate="auto", random_state=SEED, max_iter=1500).fit_transform(Xp)
        plot_2d(Y2, labels, str(out_subdir / f"{tag}_tsne_2d.png"),
                f"t-SNE 2D — {tag} (PCA{pca_dim}, pool={POOLING})")

        Y3 = TSNE(n_components=3, perplexity=perplex, init="pca",
                   learning_rate="auto", random_state=SEED, max_iter=1500).fit_transform(Xp)
        plot_3d(Y3, labels, str(out_subdir / f"{tag}_tsne_3d.png"),
                f"t-SNE 3D — {tag} (PCA{pca_dim}, pool={POOLING})")
        print(f"  [OK] {tag} t-SNE plots saved")

    np.save(out_subdir / f"{tag}_embeddings.npy", X)
    with open(out_subdir / f"{tag}_labels.json", "w") as f:
        json.dump(labels, f)
    print(f"  [OK] {tag} embeddings & labels saved")

## 1. GNN Baseline (not pretrained)

Load the end-to-end GNN baseline model and extract graph-level embeddings from the test set.

In [ ]:
# Build args for GNN baseline using the same config as the original embedding notebook
gnn_parser = configargparse.ArgumentParser(
    allow_abbrev=False,
    default_config_files=["../configs/finetune/finetune_ALL_200/finetune_ALL_200_gnn_baseline.yml"],
)
gnn_parser.add_argument("--encoder_path", type=str, default=None)
gnn_parser.add_argument("--devices", type=str, default=DEVICE_ID)
gnn_parser.add_argument("--batch_size", type=int, default=BATCH_SIZE)
gnn_parser.add_argument("--num_workers", type=int, default=0)
gnn_parser.add_argument("--seed", type=int, default=SEED)
gnn_parser.add_argument("--finetune_mode", type=str, default="full")

d = gnn_parser.add_argument_group("data")
d.add_argument("--use_bipartite_graphs", action="store_true")
d.add_argument("--problems", type=str, nargs="+",
               default=[ProblemClass.INDEPENDANT_SET, ProblemClass.COMBINATORIAL_AUCTION,
                        ProblemClass.CAPACITATED_FACILITY_LOCATION, ProblemClass.SET_COVER])
d.add_argument("--is_sizes", type=int, nargs="+", default=[200])
d.add_argument("--ca_sizes", type=int, nargs="+", default=[100])
d.add_argument("--sc_sizes", type=int, nargs="+", default=[200])
d.add_argument("--cfl_sizes", type=int, nargs="+", default=[200])
d.add_argument("--rnd_sizes", type=int, nargs="+", default=[200])
d.add_argument("--n_instances", type=int, default=35000)
d.add_argument("--data_root", type=str, default="../data/instances")
d.add_argument("--val_split", type=float, default=0.15)

gnn_args, _ = gnn_parser.parse_known_args([])
GNNPolicy.add_args(gnn_parser)
gnn_args, _ = gnn_parser.parse_known_args([])
gnn_args.encoder_path = gnn_baseline_path
print("GNN baseline args:", gnn_args)

In [ ]:
# Load data (same data as the pretrained embedding notebook)
train_loader, valid_loader, test_loader, N_max, M_max = build_dataloaders(
    gnn_args, None, None, for_pretraining=True
)

In [ ]:
# Load GNN baseline model
gnn_model = GNNPolicy(gnn_args).to(device)
pkg = torch.load(gnn_args.encoder_path, map_location="cpu")
if isinstance(pkg, dict):
    state = pkg.get("model", pkg.get("state_dict", pkg))
    gnn_model.load_state_dict(state, strict=True)
else:
    gnn_model.load_state_dict(pkg, strict=True)
gnn_model = gnn_model.to(device)
gnn_model.eval()
print(f"GNN baseline loaded from: {gnn_args.encoder_path}")

In [ ]:
# Extract GNN baseline embeddings
gnn_by_class: Dict[str, List[np.ndarray]] = defaultdict(list)

with torch.no_grad():
    for batch in tqdm(test_loader, desc="GNN baseline"):
        base = batch["base"].to(device, non_blocking=True)
        embeddings = gnn_model.embed([base])[0]
        g_emb = pooled_graph_embeddings(embeddings, base, POOLING)

        sample_paths = getattr(base, "sample_path", None)
        if sample_paths is None:
            raise RuntimeError("Batch has no sample_path attribute")

        for i, sp in enumerate(sample_paths):
            cls = infer_problem_class(sp)
            if cls in PROBLEM_CLASSES and len(gnn_by_class[cls]) < MAX_GRAPHS_PER_CLASS:
                gnn_by_class[cls].append(g_emb[i].detach().cpu().numpy())

for cls, vecs in gnn_by_class.items():
    print(f"  GNN baseline — {cls}: {len(vecs)} samples")

In [ ]:
# Reduce and plot GNN baseline
gnn_embs, gnn_labels = [], []
for cls in PROBLEM_CLASSES:
    vecs = gnn_by_class.get(cls, [])
    if not vecs:
        print(f"  [WARN] No samples for {cls}")
        continue
    gnn_embs.extend(vecs)
    gnn_labels.extend([cls] * len(vecs))
    print(f"  Using {len(vecs)} graphs for {cls}")

X_gnn = np.stack(gnn_embs, axis=0)
print(f"GNN baseline: {X_gnn.shape[0]} graphs, emb dim {X_gnn.shape[1]}")
reduce_and_plot(X_gnn, gnn_labels, "gnn_baseline", out_dir)

# Free memory
del gnn_model, gnn_by_class, gnn_embs
torch.cuda.empty_cache()

## 2. Split Architecture (not pretrained)

Load the split block model and extract graph-level embeddings. The split model processes pre-partitioned subgraphs, so it uses `SplitInstanceDataset` instead of the regular `GraphDataset`.

In [ ]:
# Build args for split model
split_parser = configargparse.ArgumentParser(
    allow_abbrev=False,
    default_config_files=["../configs/finetune_splits/finetune_split_CA_200_RAW_h0.yml"],
)
split_parser.add_argument("--encoder_path", type=str, default=None)
split_parser.add_argument("--pretrain_checkpoint", type=str, default=None)
split_parser.add_argument("--devices", type=str, default=DEVICE_ID)
split_parser.add_argument("--batch_size", type=int, default=32)
split_parser.add_argument("--num_workers", type=int, default=0)
split_parser.add_argument("--seed", type=int, default=SEED)
split_parser.add_argument("--finetune_mode", type=str, default="full")
split_parser.add_argument("--n_instances", type=int, default=None)
split_parser.add_argument("--max_owned_nodes", type=int, default=None)

d = split_parser.add_argument_group("data")
d.add_argument("--problems", type=str, nargs="+", default=["CA"])
d.add_argument("--is_sizes", type=int, nargs="+", default=[200])
d.add_argument("--ca_sizes", type=int, nargs="+", default=[100])
d.add_argument("--sc_sizes", type=int, nargs="+", default=[200])
d.add_argument("--cfl_sizes", type=int, nargs="+", default=[200])
d.add_argument("--rnd_sizes", type=int, nargs="+", default=[200])
d.add_argument("--data_root", type=str, default="../data/instances/milp/finetune")
d.add_argument("--val_split", type=float, default=0.15)

split_args, _ = split_parser.parse_known_args([])
SplitBlockBiJepaPolicy.add_args(split_parser)
split_args, _ = split_parser.parse_known_args([])
print("Split model args:", split_args)

In [ ]:
# Build split data loaders from precomputed split .pt files
def collect_split_dirs(args, split: str) -> List[Path]:
    size_cfg = {
        "IS": args.is_sizes, "CA": args.ca_sizes,
        "SC": args.sc_sizes, "CFL": args.cfl_sizes, "RND": args.rnd_sizes,
    }
    data_root = Path(args.data_root)
    if args.num_blocks is not None:
        split_variant = f"halo-{args.halo_hops}-blocks-{args.num_blocks}"
    else:
        split_variant = f"halo-{args.halo_hops}-nodes-{args.max_owned_nodes}"
    dirs = []
    for problem in args.problems:
        for size in size_cfg.get(problem, []):
            d = data_root / problem / "splits" / split_variant / split / str(size)
            if d.exists():
                dirs.append(d)
                print(f"  Found: {d}")
            else:
                print(f"  [WARN] Missing: {d}")
    return dirs

print("Test split dirs:")
test_dirs = collect_split_dirs(split_args, "test")
test_ds = SplitInstanceDataset(roots=test_dirs)
split_test_loader = DataLoader(
    test_ds, batch_size=split_args.batch_size,
    shuffle=False, num_workers=0, collate_fn=split_instance_collate,
)
print(f"Split test set: {len(test_ds)} instances, {len(split_test_loader)} batches")

In [ ]:
# Load split model
split_model = SplitBlockBiJepaPolicy(split_args).to(device)
pkg = torch.load(split_model_path, map_location="cpu")
if isinstance(pkg, dict):
    state = pkg.get("model", pkg.get("state_dict", pkg))
    split_model.load_state_dict(state, strict=True)
else:
    split_model.load_state_dict(pkg, strict=True)
split_model = split_model.to(device)
split_model.eval()
print(f"Split model loaded from: {split_model_path}")

In [ ]:
# Extract split model embeddings
split_by_class: Dict[str, List[np.ndarray]] = defaultdict(list)

with torch.no_grad():
    for batch in tqdm(split_test_loader, desc="Split model"):
        g_emb, names = pooled_split_embeddings(split_model, batch, POOLING)
        for i, name in enumerate(names):
            cls = infer_problem_class(name)
            if cls in PROBLEM_CLASSES and len(split_by_class[cls]) < MAX_GRAPHS_PER_CLASS:
                split_by_class[cls].append(g_emb[i].detach().cpu().numpy())

for cls, vecs in split_by_class.items():
    print(f"  Split model — {cls}: {len(vecs)} samples")

In [ ]:
# Reduce and plot split model
split_embs, split_labels = [], []
for cls in PROBLEM_CLASSES:
    vecs = split_by_class.get(cls, [])
    if not vecs:
        print(f"  [WARN] No samples for {cls}")
        continue
    split_embs.extend(vecs)
    split_labels.extend([cls] * len(vecs))
    print(f"  Using {len(vecs)} graphs for {cls}")

if split_embs:
    X_split = np.stack(split_embs, axis=0)
    print(f"Split model: {X_split.shape[0]} graphs, emb dim {X_split.shape[1]}")
    reduce_and_plot(X_split, split_labels, "split_raw", out_dir)
else:
    print("[WARN] No split embeddings collected — check data paths and problem classes")

del split_model, split_by_class, split_embs
torch.cuda.empty_cache()

## Summary

All plots and embeddings saved to `results/finetuned_embeddings/`:
- `gnn_baseline_{pca,tsne}_{2d,3d}.png` — GNN baseline (end-to-end, no pretraining)
- `split_raw_{pca,tsne}_{2d,3d}.png` — Split architecture (block-decomposed, no pretraining)
- `*_embeddings.npy` / `*_labels.json` — raw data for further analysis

In [ ]:
print("Output files:")
for f in sorted(out_dir.iterdir()):
    print(f"  {f.name}")